In [ ]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from pathlib import Path
from pyteomics.fasta import FASTA
import requests
import sys
import re
import mhcgnomes

This notebook is intended to give an overview of the data processing. We provide the processed data, for most parts, but not the unprocessed data and also not the PCI-DB.

# MHC/HLA-DB

In [ ]:
fasta = []
for record in FASTA('data/hla_prot.fasta'):
    hla = 'HLA-' + record.description.split()[1]
    fasta.append([re.sub(r"[A-Za-z]+$", "", hla), record.sequence])
    
for record in FASTA('data/MHC_prot.fasta'):
    mhc = record.description.split()[1]
    fasta.append([re.sub(r"[A-Za-z]+$", "", mhc), record.sequence])

mhc_prot = pd.DataFrame(fasta, columns=['allele', 'sequence']).dropna()

url = 'https://www.ebi.ac.uk/cgi-bin/ipd/api/allele?query=and(eq(confirmation_status.confirmed,true),eq(sequence_status.full,true))'
next_page = ''
cd_df = []
i = 0
while True:
    response = requests.get(url + next_page)
    response.raise_for_status()
    cd_df.append(pd.DataFrame(response.json()['data']))
    i += len(response.json()['data'])
    sys.stdout.write(f"\rProcessing {i + 1}/{response.json()['meta']['total']}")
    sys.stdout.flush()
    if response.json()['meta']['next'] is not None:
        next_page = '&' + response.json()['meta']['next'].split('&')[-1]
    else:
        break

cd_df = pd.concat(cd_df)        
cd_df['name'] = 'HLA-' + cd_df['name']

mhc_prot = pd.merge(mhc_prot, cd_df[['name', 'sequence_status.full']], left_on='allele', right_on='name', how='left')
mhc_prot = mhc_prot.loc[
    (mhc_prot['allele'].str.contains('HLA') & mhc_prot['sequence_status.full']) |
    ~mhc_prot['allele'].str.contains('HLA')
].drop(['name', 'sequence_status.full'], axis=1)

mhc_prot = mhc_prot.loc[mhc_prot['sequence'].str.len() >= 200]

mhc_prot['allele'] = mhc_prot['allele'].str.split(':').str[:2].str.join(':')
mhc_prot = mhc_prot.groupby('allele')['sequence'] \
    .apply(lambda x: sorted(x, key=len)[-1]) \
    .reset_index()

mhc_prot = mhc_prot.drop_duplicates(keep=False, subset='sequence')  # we don't know to which allele a sequence belongs if it is duplicated

mhc_prot

In [ ]:
mhc_prot.to_parquet('data/mhc_db.parquet', index=False)

# BA-DB

In [ ]:
ba_df = pd.concat([pd.read_csv(path, sep=' ', header=None) for path in Path('data/NetMHCpan_train').glob('*_ba')])
ba_df.columns = ['peptide', 'ba', 'allele']
ba_df = ba_df.drop_duplicates()

# convert allele names to standard format
new_alleles = []
for alleles in tqdm(ba_df['allele'].unique()):
    _alleles = []
    for allele in re.split(',', alleles):
        if allele != '' and mhcgnomes.parse(allele, raise_on_error=False) is not None:
            allele = mhcgnomes.parse(allele).to_string()
            _alleles.append(allele)
    new_alleles.append([alleles, ','.join(_alleles)])

ba_df = pd.merge(
    ba_df,
    pd.DataFrame(new_alleles, columns=['allele', 'new_allele'])
)
ba_df = ba_df.loc[ba_df['new_allele'] != ''].drop('allele', axis=1)
ba_df.columns = ['peptide', 'ba', 'allele']

ba_df['ba'] = ba_df['ba'].astype(np.float32)

ba_df

In [ ]:
ba_df.to_parquet('data/ba_db.parquet', index=False)

# EL-DB

In [ ]:
allelelist = pd.read_csv('data/NetMHCpan_train/allelelist', sep='\t', header=None)
allelelist.columns = ['name', 'allele']
allelelist = dict(allelelist.values)

el_df = pd.concat([pd.read_csv(path, sep=' ', header=None) for path in Path('data/NetMHCpan_train').glob('*_el')])
el_df.columns = ['peptide', 'el', 'allele']
el_df['allele'] = el_df['allele'].apply(lambda x: allelelist[x] if x in allelelist else x)
el_df['source'] = 'netmhcpan_el'

# convert allele names to standard format
new_alleles = []
for alleles in tqdm(el_df['allele'].unique()):
    _alleles = []
    for allele in re.split(',', alleles):
        if allele != '' and mhcgnomes.parse(allele, raise_on_error=False) is not None:
            allele = mhcgnomes.parse(allele).to_string()
            _alleles.append(allele)
    new_alleles.append([alleles, ','.join(_alleles)])

el_df = pd.merge(
    el_df,
    pd.DataFrame(new_alleles, columns=['allele', 'new_allele'])
)
el_df = el_df.loc[el_df['new_allele'] != ''].drop('allele', axis=1)
el_df.columns = ['peptide', 'el', 'source', 'allele']

el_df['peptide'] = el_df['peptide'].str.replace('[a-z]', '', regex=True)
el_df['el'] = el_df['el'].astype(np.int32)
el_df

In [ ]:
el_df.to_parquet('data/netmhcpan_el.parquet', index=False)

In [ ]:
mono_db = pd.read_csv('data/250109_latest_data/monoDB_EL.tsv', sep='\t')
mono_db.columns = ['peptide', 'allele', 'source']

# convert allele names to standard format
new_alleles = []
for alleles in tqdm(mono_db['allele'].unique()):
    _alleles = []
    for allele in re.split(',', alleles):
        if allele != '' and mhcgnomes.parse(allele, raise_on_error=False) is not None:
            allele = mhcgnomes.parse(allele).to_string()
            # if allele in mhc_prot['allele'].values:
            _alleles.append(allele)
    new_alleles.append([alleles, ','.join(_alleles)])

mono_db = pd.merge(
    mono_db,
    pd.DataFrame(new_alleles, columns=['allele', 'new_allele'])
)
mono_db = mono_db.loc[mono_db['new_allele'] != ''].drop('allele', axis=1)
mono_db.columns = ['peptide', 'source', 'allele']

mono_db

In [ ]:
mhcmotif = pd.concat([
    pd.read_csv('data/data_classI_all_peptides_20260206.txt', sep='\t'),
    pd.read_csv('data/data_classII_MS_Peptides_all_peptides_20260206.txt', sep='\t').iloc[:, :-1]
])
mhcmotif.columns = ['allele', 'peptide']
mhcmotif['allele'] = mhcmotif['allele'].str.replace('__', ',').str.replace(',reverse', '')
mhcmotif['source'] = 'mhcmotifatlas'

# Gaga-BLB is not included right now, but maybe in the future
mhcmotif.loc[mhcmotif['allele'].str.contains('Gaga'), 'allele'] = 'Gaga-' + mhcmotif.loc[mhcmotif['allele'].str.contains('Gaga'), 'allele'].str.split('_').str[1] + \
    '*' + mhcmotif.loc[mhcmotif['allele'].str.contains('Gaga'), 'allele'].str.split('_').str[2:].str.join(':')
mhcmotif.loc[mhcmotif['allele'].str.contains('BoLA'), 'allele'] = 'BoLA-' + mhcmotif.loc[mhcmotif['allele'].str.contains('BoLA'), 'allele'].str.split('_').str[1] + \
    '*' + mhcmotif.loc[mhcmotif['allele'].str.contains('BoLA'), 'allele'].str.split('_').str[2:].str.join(':')

# convert allele names to standard format
new_alleles = []
for alleles in tqdm(mhcmotif['allele'].unique()):
    _alleles = []
    for allele in re.split(',', alleles):
        if allele.startswith('D'):
            allele = allele.split('_')[0] + '*' + ':'.join(allele.split('_')[1:])
        if allele != '' and mhcgnomes.parse(allele, raise_on_error=False) is not None:
            allele = mhcgnomes.parse(allele).to_string()
            # if allele in mhc_prot['allele'].values:
            _alleles.append(allele)
    new_alleles.append([alleles, ','.join(_alleles)])

mhcmotif = pd.merge(
    mhcmotif,
    pd.DataFrame(new_alleles, columns=['allele', 'new_allele'])
)
mhcmotif = mhcmotif.loc[mhcmotif['new_allele'] != ''].drop('allele', axis=1)
mhcmotif.columns = ['peptide', 'source', 'allele']

mhcmotif

In [ ]:
# mhcmotifatlas I & II
# remove duplicates in peptide, allele combination

In [ ]:
el_db = pd.concat([el_df, mono_db, mhcmotif])
el_db['peptide'] = el_db['peptide'].str.replace('[a-z]', '', regex=True)
el_db['allele'] = el_db['allele'].str.replace(',,', ',', regex=True).str.strip(',')
el_db.drop_duplicates(inplace=True, subset=['peptide', 'allele'], keep='first')
el_db['el'] = el_db['el'].fillna(1).astype(np.int32)
el_db

In [ ]:
el_db.to_parquet('data/el_db.parquet', index=False)

## Lookup

In [ ]:
el_db = pd.read_parquet('data/el_db.parquet')
el_db

In [ ]:
el_db = el_db.loc[
    (el_db['el'] == 1) & ~el_db['allele'].str.contains(',') & el_db['allele'].str.contains('HLA-[ABC]', regex=True)
    ].drop(['el', 'source'], axis=1)
el_db = el_db.groupby('peptide').filter(lambda x: x['allele'].nunique() == 1)
el_db['locus'] = el_db['allele'].str[4]
el_db

In [ ]:
el_db.to_parquet('data/lookup_db.parquet', index=False)

# PCI-DB

In [ ]:
# !This will not run without the corresponding data

pci_db = pd.read_csv('data/tue_db_050325_default_fix_BD.tsv', sep='\t')
# Add new data from tuedb_extention.tsv

hnsc_df = pd.read_table('data/260203_hnscc_data_for_immunotype.tsv')
hnsc_df['all_hla_alleles_donor'] = hnsc_df['alleles']
hnsc_df['peptide_sequence_mods'] = hnsc_df['peptide']
hnsc_df['biological_material_name'] = hnsc_df['tissue']

pci_db = pd.concat([
    pci_db, 
    pd.read_table('data/250109_latest_data/tuedb_extention.tsv'),
    hnsc_df  # Added February 3, 2026
])


# Replace all_hla_alleles_donor of 29-14 with 637-13 and vice versa since they seem to be swapped in the original data
tmp_29_14 = pci_db.loc[pci_db['all_hla_alleles_donor'] == '29-14', 'all_hla_alleles_donor'].unique()
tmp_637_13 = pci_db.loc[pci_db['all_hla_alleles_donor'] == '637-13', 'all_hla_alleles_donor'].unique()
pci_db.loc[pci_db['all_hla_alleles_donor'] == '637-13', 'all_hla_alleles_donor'] = tmp_29_14
pci_db.loc[pci_db['all_hla_alleles_donor'] == '29-14', 'all_hla_alleles_donor'] = tmp_637_13

# Keep only relevant columns and drop rows with NaN in key columns
pci_db = pci_db[['peptide_sequence_mods', 'all_hla_alleles_donor', 'mhc_class', 'sample_code', 'donor_code', 'biological_material_name', 'qbic_project_code']]
pci_db = pci_db.dropna(subset=['peptide_sequence_mods', 'all_hla_alleles_donor', 'donor_code'])
pci_db.columns = ['peptide', 'alleles', 'mhc_class', 'sample_code', 'donor_code', 'tissue', 'qbic_project_code']
pci_db['peptide'] = pci_db['peptide'].str.replace(r'\(\w*\)', '', regex=True)
pci_db = pci_db.drop_duplicates()

### Dec 25
pxd037270_json = pd.read_json('data/PXD037270.json', orient='index')
pxd037270_json.columns = ['alleles']
pxd037270_json = pxd037270_json.reset_index(names='donor_code').set_index('donor_code').to_dict()['alleles']

pci_db['alleles'] = pci_db['alleles']\
    .str.replace('C*01:02DPB1*04:01', 'C*01:02;DPB1*04:01')\
    .str.replace('C*04:01DPB1*02:01', 'C*04:01;DPB1*02:01')\
    .str.replace('C*05:01DPB1*03:01 ', 'C*05:01;DPB1*03:01')\
    .str.replace('C*07:04DPB1*04:01', 'C*07:04;DPB1*04:01')\
    .str.replace('C*16:01DPB1*04:01', 'C*16:01;DPB1*04:01')\
    .str.replace(' ', '')
pci_db['alleles'] = pci_db.apply(lambda x: pxd037270_json[x['donor_code']] if x['donor_code'] in pxd037270_json else x['alleles'], axis=1)


pci_db['alleles'] = pci_db['alleles'].apply(
    lambda x: ','.join([
        'HLA-' + ':'.join(allele.split(':')[:2]) for allele in x.split(';') 
        if allele[-1] != 'N' # filter out null alleles
        and allele.count(':') != 0 # filter out 2 digit alleles
        and allele != ''
    ])
) # cut down to protein level info

pci_db = pci_db.drop_duplicates(subset=['peptide', 'donor_code', 'sample_code'], keep='first') # remove duplicate combinations
pci_db = pci_db.loc[pci_db['alleles'] != '']
pci_db = pci_db.loc[
    ~((pci_db['mhc_class'].values == 'I') & np.min([pci_db['alleles'].str.count(f'HLA-{l}') == 0 for l in ['A', 'B', 'C']], axis=0))
] # filter out class I peptides where the donor has no class I typing available

pci_db = pci_db.loc[
    ~((pci_db['mhc_class'].values == 'II') & ~pci_db['alleles'].str.contains('D'))
] # filter out class II peptides where the donor has no class II typing available

pci_db['sample_code'] = pci_db['sample_code'].astype(str)

pci_db_samples = pci_db.drop('peptide', axis=1).drop_duplicates().copy()
pci_db_samples['alleles'] = [
    ','.join([x for x in y if ('HLA-D' in x and c == 'II') or (not 'HLA-D' in x and c == 'I')]) 
    for y, c in zip(pci_db_samples['alleles'].str.split(',').values, pci_db_samples['mhc_class'])
]
pci_db_samples.columns = pci_db_samples.columns.str.replace('alleles', 'allele')
pci_db = pd.merge(pci_db.drop('alleles', axis=1), pci_db_samples)

pci_db

In [ ]:
donor_df = pci_db[['allele', 'donor_code']].drop_duplicates()
donor_df['allele'] = donor_df['allele'].str.split(',')
donor_df = donor_df.explode(column='allele').drop_duplicates()
donor_df['label'] = 1
donor_df = donor_df.pivot(columns='allele', index='donor_code', values='label').fillna(0)
homo_df = donor_df.T.groupby(donor_df.T.index.str.split('*').str[0]).apply(lambda x: (x.sum(axis=0) == 1) * 1).T
homo_df.columns = [f'{c}*homozygous' for c in homo_df.columns]
donor_df = donor_df.loc[:, donor_df.columns.isin(mhc_prot['allele'].values)]
donor_df = pd.concat([donor_df, homo_df], axis=1).astype(int)
donor_df = donor_df + donor_df.T.groupby(donor_df.T.index.str.split('*').str[0]).transform(lambda x: (x.sum(axis=0) == 0) * -1).T
donor_df = donor_df.loc[:, ~donor_df.columns.str.contains('D')]
donor_df = donor_df.loc[donor_df.sum(axis=1) == 6]
donor_df

In [ ]:
pci_db.to_parquet('data/pci_db.parquet', index=False)
donor_df.to_parquet('data/pci_db_donors.parquet')